# Chess data collection

Reconstructed documentation of the collection procedure, using the archived player-list snapshot and seed 2950. These collection cells have not been executed in this copy. Use the archived raw CSV to reproduce Phase 2 without recollecting. Archive availability may change if this code is run later.

In [ ]:
import json
import random
import requests
import pandas as pd

# Use the archived player-list snapshot, not a new list of different accounts.
base = "https://raw.githubusercontent.com/Toyman123/info2950-project/main/phase2"
headers = {"User-Agent": "INFO2950-Chess-Project"}
response = requests.get(base + "/data/raw/player_list_raw.json", timeout=60)
response.raise_for_status()
players = sorted(set(response.json()["players"]))
random.Random(2950).shuffle(players)
players = players[:300]
rows = []
archive_log = []

# This downloads source archives again. Use the saved raw CSV for analysis.
with open("original_game_data.json", "w") as raw_file:
    raw_file.write("[")
    first_response = True
    for username in players:
        for month in [7, 8, 9]:
            url = f"https://api.chess.com/pub/player/{username}/games/2026/{month:02d}"
            response = requests.get(url, headers=headers, timeout=60)
            archive_log.append({"username": username, "month": month,
                                "url": url, "status": response.status_code})
            if response.status_code in [404, 410]:
                continue
            response.raise_for_status()
            if not first_response:
                raw_file.write(",")
            raw_file.write(response.text)
            first_response = False
            for game in response.json().get("games", []):
                white = game.get("white", {})
                black = game.get("black", {})
                rows.append({
                    "sampled_player": username, "archive_month": month,
                    "game_url": game.get("url"),
                    "white_username": white.get("username"),
                    "black_username": black.get("username"),
                    "white_rating": white.get("rating"),
                    "black_rating": black.get("rating"),
                    "white_result": white.get("result"),
                    "black_result": black.get("result"),
                    "rated": game.get("rated"), "rules": game.get("rules"),
                    "time_class": game.get("time_class"),
                    "time_control": game.get("time_control"),
                    "end_time": game.get("end_time"),
                    "opening_url": game.get("eco"), "pgn": game.get("pgn")
                })
    raw_file.write("]")
pd.DataFrame(rows).to_csv("chess_games_raw.csv", index=False)
pd.DataFrame(archive_log).to_csv("archive_log.csv", index=False)
pd.DataFrame({"username": players}).to_csv("selected_players.csv", index=False)
print("Rows downloaded:", len(rows))
